In [61]:
from pathlib import Path
import pandas as pd
import numpy as np


In [62]:
BASE_DIR = Path(r"C:\Users\nelid\Documents\Kaggle Competitions\Store Sales Competition\store-sales")
RAW_DIR       = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"


In [63]:
# Rohdaten laden

train        = pd.read_csv(RAW_DIR / "train.csv",            parse_dates=["date"])
test         = pd.read_csv(RAW_DIR / "test.csv",             parse_dates=["date"])
stores       = pd.read_csv(RAW_DIR / "stores.csv")
oil          = pd.read_csv(RAW_DIR / "oil.csv",              parse_dates=["date"])
holidays     = pd.read_csv(RAW_DIR / "holidays_events.csv",  parse_dates=["date"])
transactions = pd.read_csv(RAW_DIR / "transactions.csv",     parse_dates=["date"])

In [64]:
# Store-Features mergen
train = train.merge(stores, on="store_nbr", how="left")
test  = test.merge(stores,  on="store_nbr", how="left")

print("Train:", train.shape, "| Test:", test.shape)
print("Test-Datum-Range:", test["date"].min(), "→", test["date"].max())
print("Letztes Train-Datum:", train["date"].max())
# Test beginnt 16 Tage nach dem letzten Train-Datum → Lags >= 16 sind sicher

Train: (3000888, 10) | Test: (28512, 9)
Test-Datum-Range: 2017-08-16 00:00:00 → 2017-08-31 00:00:00
Letztes Train-Datum: 2017-08-15 00:00:00


In [65]:
# Zeitfeatures wie im ersten Versuch

def add_time_features(df):
    df = df.copy()
    df["year"]       = df["date"].dt.year
    df["month"]      = df["date"].dt.month
    df["day"]        = df["date"].dt.day
    df["dayofweek"]  = df["date"].dt.dayofweek
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
    df["dayofyear"]  = df["date"].dt.dayofyear
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["quarter"]    = df["date"].dt.quarter

    # Zyklische Encodings
    df["month_sin"]  = np.sin(2 * np.pi * df["month"]     / 12)
    df["month_cos"]  = np.cos(2 * np.pi * df["month"]     / 12)
    df["dow_sin"]    = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * df["dayofweek"] / 7)
    df["woy_sin"]    = np.sin(2 * np.pi * df["weekofyear"]/ 52)
    df["woy_cos"]    = np.cos(2 * np.pi * df["weekofyear"]/ 52)
    df["doy_sin"]    = np.sin(2 * np.pi * df["dayofyear"] / 365)
    df["doy_cos"]    = np.cos(2 * np.pi * df["dayofyear"] / 365)
    return df

train = add_time_features(train)
test  = add_time_features(test)

# ─────────────────────────────────────────────────────────────────────────────
# 7. NEU: Payday-Features
#    Gehälter im öffentlichen Sektor: 15. und letzter Tag des Monats
# ─────────────────────────────────────────────────────────────────────────────
def add_payday_features(df):
    df = df.copy()
    day      = df["date"].dt.day
    last_day = (df["date"] + pd.offsets.MonthEnd(0)).dt.day

    df["is_payday_15th"]  = (day == 15).astype(int)
    df["is_payday_last"]  = (day == last_day).astype(int)
    df["is_payday"]       = ((df["is_payday_15th"] == 1) | (df["is_payday_last"] == 1)).astype(int)

    # Tage bis zum nächsten Zahltag
    days_until_15   = (15 - day).clip(lower=0)
    days_until_last = (last_day - day).clip(lower=0)
    df["days_until_payday"] = np.minimum(days_until_15, days_until_last)

    # Tage seit dem letzten Zahltag
    days_since_15 = (day - 15).clip(lower=0)
    # Vor dem 15.: Abstand zum letzten Tag des Vormonats
    df["days_since_payday"] = np.where(day >= 15, days_since_15, day - 1)
    # Sonderfall: nach letztem Tag → 0
    df["days_since_payday"] = np.where(
        df["is_payday_last"] == 1, 0, df["days_since_payday"]
    )

    # Kaufsog-Fenster: ±3 Tage um Zahltag
    df["is_payday_window"] = (
        (df["days_until_payday"] <= 3) | (df["days_since_payday"] <= 3)
    ).astype(int)

    return df

train = add_payday_features(train)
test  = add_payday_features(test)



In [66]:
# Payday features
# Gehälter im öffentlichen Sektor werden am 15. und am
# letzten Tag des Monats ausgezahlt → erhöhte Kaufkraft

def add_payday_features(df):
    df = df.copy()

    day      = df["date"].dt.day
    # Letzter Tag des jeweiligen Monats
    last_day = (df["date"] + pd.offsets.MonthEnd(0)).dt.day

    is_15th    = (day == 15).astype(int)
    is_lastday = (day == last_day).astype(int)

    df["is_payday"]       = ((is_15th == 1) | (is_lastday == 1)).astype(int)
    df["is_payday_15th"]  = is_15th
    df["is_payday_last"]  = is_lastday

    # Tage bis zum nächsten Zahltag
    days_until_15   = (15 - day).clip(lower=0)
    days_until_last = (last_day - day).clip(lower=0)
    df["days_until_payday"] = np.minimum(days_until_15, days_until_last)

    # Tage seit dem letzten Zahltag
    days_since_15   = (day - 15).clip(lower=0)
    days_since_last = (day - last_day).clip(lower=0)   # immer 0 oder negativ→0
    # für Tage VOR dem 15.: Abstand zum letzten Tag des Vormonats
    days_since_prev_last = day - 1   # vereinfacht (1. = 0, 2. = 1 …)
    df["days_since_payday"] = np.where(
        day >= 15,
        np.minimum(days_since_15, days_since_last),
        days_since_prev_last
    )

    # Woche um den Zahltag herum (±3 Tage) – Kaufsog vor/nach Zahltag
    df["is_payday_window"] = (
        (df["days_until_payday"] <= 3) | (df["days_since_payday"] <= 3)
    ).astype(int)

    return df

train = add_payday_features(train)
test  = add_payday_features(test)

# Sanity-Check
print("Zahltage im Train-Set:", train["is_payday"].sum() // train["store_nbr"].nunique() // train["family"].nunique())
print(train[["date","is_payday","days_since_payday","days_until_payday","is_payday_window"]]
      .drop_duplicates("date").query("is_payday==1").head(6))


Zahltage im Train-Set: 111
             date  is_payday  days_since_payday  days_until_payday  \
24948  2013-01-15          1                  0                  0   
53460  2013-01-31          1                  0                  0   
80190  2013-02-15          1                  0                  0   
103356 2013-02-28          1                  0                  0   
130086 2013-03-15          1                  0                  0   
158598 2013-03-31          1                  0                  0   

        is_payday_window  
24948                  1  
53460                  1  
80190                  1  
103356                 1  
130086                 1  
158598                 1  


In [67]:
# Erdbeben-Features
# Magnitude 7.8 Erdbeben am 16. April 2016 in Ecuador.
# Bevölkerung kaufte/spendete Wasser & Grundbedarfsgüter
# → starker Einfluss auf Supermarkt-Umsätze für mehrere Wochen

EARTHQUAKE_DATE = pd.Timestamp("2016-04-16")

def add_earthquake_features(df, eq_date=EARTHQUAKE_DATE, impact_weeks=8):
    df = df.copy()

    delta_days = (df["date"] - eq_date).dt.days

    # Tag des Erdbebens
    df["is_earthquake_day"] = (df["date"] == eq_date).astype(int)

    # Tage seit Erdbeben (negativ = vor Erdbeben → wird als 0 behandelt)
    df["days_since_earthquake"] = delta_days.clip(lower=0)

    # Ob wir im Nachwirkungszeitraum sind (bis impact_weeks Wochen danach)
    max_days = impact_weeks * 7
    in_impact = (delta_days >= 0) & (delta_days <= max_days)

    # Exponentiell abklingendes Signal (Halbwertszeit ~14 Tage)
    df["earthquake_impact"] = np.where(
        in_impact,
        np.exp(-delta_days.clip(lower=0) / 14.0),
        0.0
    )

    # Binäre Flags pro Woche nach dem Erdbeben (Woche 1–4)
    for w in range(1, 5):
        lo = (w - 1) * 7
        hi = w * 7
        df[f"is_post_eq_week{w}"] = (
            (delta_days >= lo) & (delta_days < hi)
        ).astype(int)

    # Wochen 5–8 zusammengefasst als "spätere Nachwirkung"
    df["is_post_eq_week5_8"] = (
        (delta_days >= 28) & (delta_days < 56)
    ).astype(int)

    return df

train = add_earthquake_features(train)
test  = add_earthquake_features(test)

# Sanity-Check: zeige die Woche direkt nach dem Erdbeben
print("\nErdbeben-Features (direkt nach 16.04.2016):")
eq_check = (train[["date","is_earthquake_day","days_since_earthquake",
                    "earthquake_impact","is_post_eq_week1","is_post_eq_week2"]]
            .drop_duplicates("date")
            .query("date >= '2016-04-14' and date <= '2016-05-05'"))
print(eq_check)




Erdbeben-Features (direkt nach 16.04.2016):
              date  is_earthquake_day  days_since_earthquake  \
2131272 2016-04-14                  0                      0   
2133054 2016-04-15                  0                      0   
2134836 2016-04-16                  1                      0   
2136618 2016-04-17                  0                      1   
2138400 2016-04-18                  0                      2   
2140182 2016-04-19                  0                      3   
2141964 2016-04-20                  0                      4   
2143746 2016-04-21                  0                      5   
2145528 2016-04-22                  0                      6   
2147310 2016-04-23                  0                      7   
2149092 2016-04-24                  0                      8   
2150874 2016-04-25                  0                      9   
2152656 2016-04-26                  0                     10   
2154438 2016-04-27                  0                     1

In [68]:
# Ölpreis-Feature, interpoliert, kein Lookahead
oil = oil.rename(columns={"dcoilwtico": "oil_price"})
oil = (
    oil.set_index("date")
    .reindex(pd.date_range(oil["date"].min(), oil["date"].max(), freq="D"))
    .rename_axis("date")
    .reset_index()
)
oil["oil_price"] = oil["oil_price"].interpolate(method="linear")
# Öl-Lags (16, 21, 28 Tage) – kein Leakage im Test
for lag in [16, 21, 28]:
    oil[f"oil_lag_{lag}"] = oil["oil_price"].shift(lag)
# Rolling Mean des Ölpreises
oil["oil_rmean_28"] = oil["oil_price"].rolling(28, min_periods=1).mean()

train = train.merge(oil.drop(columns=["oil_price"]), on="date", how="left")
test  = test.merge(oil.drop(columns=["oil_price"]),  on="date", how="left")
# Wichtig: oil_price selbst weglassen (hätte Lookahead-Problem),
# stattdessen nur die Lags verwenden



In [69]:
# Holiday Features

# Nur nationale Feiertage (type == "Holiday"), keine lokalen
nat_hol = holidays[
    (holidays["locale"] == "National") &
    (holidays["transferred"] == False)
][["date", "type", "description"]].copy()
nat_hol = nat_hol.rename(columns={"description": "holiday_name"})

# Alle Holiday-Typen als binäre Flags
hol_pivot = pd.get_dummies(
    holidays.assign(val=1)[["date","type","val"]].drop_duplicates(),
    columns=["type"], prefix="hol"
).groupby("date").max().reset_index()

# Einfache Flags
holiday_daily = (
    holidays.groupby("date")
    .agg(
        is_holiday_observed  = ("type", lambda x: int("Holiday" in x.values)),
        is_additional_holiday= ("type", lambda x: int("Additional" in x.values)),
        is_bridge_day        = ("type", lambda x: int("Bridge" in x.values)),
        is_work_day_comp     = ("type", lambda x: int("Work Day" in x.values)),
        is_transferred_flag  = ("transferred", "max"),
        holiday_names        = ("description", lambda x: "|".join(x.unique()))
    )
    .reset_index()
)
holiday_daily["is_holiday_weekend"] = (
    holiday_daily["is_holiday_observed"] &
    holiday_daily["date"].dt.dayofweek.isin([5, 6])
).astype(int)

holiday_daily.to_csv(PROCESSED_DIR / "holiday_features_daily_v2.csv", index=False)

train = train.merge(holiday_daily, on="date", how="left")
test  = test.merge(holiday_daily,  on="date", how="left")

# NaN-Füllung bei Holiday-Spalten
hol_cols = [c for c in holiday_daily.columns if c != "date"]
train[hol_cols] = train[hol_cols].fillna({"holiday_names": "none"}).fillna(0)
test[hol_cols]  = test[hol_cols].fillna({"holiday_names": "none"}).fillna(0)


In [70]:
# Transactions - Feature (nur Train – im Test unbekannt, daher Lags davon)

train = train.merge(transactions, on=["date", "store_nbr"], how="left")
test["transactions"] = np.nan  # Platzhalter, wird durch Lags ersetzt

In [71]:
# Speichere Basis-Datensatz vor Lags

train.to_csv(PROCESSED_DIR / "train_model_base_v2.csv", index=False)
test.to_csv( PROCESSED_DIR / "test_model_base_v2.csv",  index=False)
print("\nGespeichert: train_model_base_v2.csv, test_model_base_v2.csv")
print("Train shape:", train.shape)
print("Test shape: ", test.shape)


Gespeichert: train_model_base_v2.csv, test_model_base_v2.csv
Train shape: (3000888, 52)
Test shape:  (28512, 51)


- Ab hier werden Lags berechnet. Am besten auf dem vollen Datensatz


In [72]:
# Train + Test zusammenführen für konsistente Lags
test["sales"] = np.nan
full = pd.concat([train, test], axis=0).sort_values(
    ["store_nbr", "family", "date"]
).reset_index(drop=True)

grp = ["store_nbr", "family"]


In [73]:
# 9. Promotions Rolling-Features
#    Wie oft war das Produkt in den letzten N Tagen in Aktion?
#    (nur Lags >= 16, damit kein Leakage im Test)
# 
for lag in [16, 21, 28]:
    full[f"promo_lag_{lag}"] = full.groupby(grp)["onpromotion"].shift(lag)

# Promotions Rolling-Summen (basierend auf Lag-16)
for win in [7, 14, 28]:
    full[f"promo_rsum_lag16_win{win}"] = (
        full.groupby(grp)["promo_lag_16"]
        .transform(lambda s: s.rolling(win, min_periods=1).sum())
    )



In [74]:
# Promotions-Interaktion: Wie reagiert diese Serie auf Promotions?

# Historischer Promotion-Umsatz-Effekt pro Store x Family
promo_effect = (
    full[full["sales"].notna() & (full["onpromotion"] > 0)]
    .groupby(grp)["sales"]
    .mean()
    .rename("mean_sales_on_promo")
    .reset_index()
)
no_promo_effect = (
    full[full["sales"].notna() & (full["onpromotion"] == 0)]
    .groupby(grp)["sales"]
    .mean()
    .rename("mean_sales_no_promo")
    .reset_index()
)
promo_lift = promo_effect.merge(no_promo_effect, on=grp, how="left")
promo_lift["promo_lift_ratio"] = (
    promo_lift["mean_sales_on_promo"] / (promo_lift["mean_sales_no_promo"] + 1e-6)
)
full = full.merge(promo_lift[grp + ["promo_lift_ratio"]], on=grp, how="left")

In [75]:
# Transactions Rolling Features (Lag >= 16)
# Wichtig für den Test Datensatz

for lag in [16, 21, 28]:
    full[f"transactions_lag_{lag}"] = full.groupby(["store_nbr", "date"]
        )["transactions"].transform("first")  # transactions ist pro Store/Tag, nicht pro Family
    # Korrekte Berechnung: erst auf Store-Ebene shiften
    full[f"transactions_lag_{lag}"] = (
        full.groupby("store_nbr")["transactions"].shift(lag)
    )

for win in [7, 14, 28]:
    full[f"transactions_rsum_lag16_win{win}"] = (
        full.groupby("store_nbr")["transactions_lag_16"]
        .transform(lambda s: s.rolling(win, min_periods=1).mean())
    )

# 

In [76]:
# Sales Lag- und Rolling-Features (NUR Lags >= 16!)
#     Wichtigste Features – kein Leakage im Test-Set
# 

LAGS           = [16, 21, 28, 35, 42, 56, 90, 364, 365, 371]
ROLLING_WINDOWS = [7, 14, 28, 56, 90]

# Zuerst: Lag-Spalten anlegen
for lag in LAGS:
    full[f"sales_lag_{lag}"] = full.groupby(grp)["sales"].shift(lag)

for lag in LAGS:
    lag_col = f"sales_lag_{lag}"
    for win in ROLLING_WINDOWS:
        full[f"sales_rmean_lag{lag}_win{win}"] = (
            full.groupby(grp)[lag_col]
            .transform(lambda s: s.rolling(win, min_periods=1).mean())
        )

# Rolling Std (Volatilität) auf Basis Lag-16
for win in [14, 28]:
    full[f"sales_rstd_lag16_win{win}"] = (
        full.groupby(grp)["sales_lag_16"]
        .transform(lambda s: s.rolling(win, min_periods=1).std().fillna(0))
    )

# Rolling Max/Min (Spitzen erkennen)
for win in [14, 28]:
    full[f"sales_rmax_lag16_win{win}"] = (
        full.groupby(grp)["sales_lag_16"]
        .transform(lambda s: s.rolling(win, min_periods=1).max())
    )
    full[f"sales_rmin_lag16_win{win}"] = (
        full.groupby(grp)["sales_lag_16"]
        .transform(lambda s: s.rolling(win, min_periods=1).min())
    )

# 



In [77]:
# Trend-Feature: Vergleich kurzfristiger vs. langfristiger Mittelwert
# Zeigt ob eine Serie gerade im Auf- oder Abwaertstrend ist
# Kurz- vs. Langfrist-Verhaeltnis (Momentum)

full["sales_momentum"] = (
    full["sales_rmean_lag16_win7"] / (full["sales_rmean_lag16_win90"] + 1e-6)
)

# Vorjahres-Verhaeltnis (YoY-Wachstum)
full["sales_yoy_ratio"] = (
    full["sales_rmean_lag16_win28"] / (full["sales_rmean_lag364_win28"] + 1e-6)
)


In [78]:
#  Target Encoding: historischer Mittelwert pro Store×Family
#     NUR aus Train berechnen → auf Test anwenden (kein Leakage)

train_mask = full["sales"].notna()

target_enc = (
    full[train_mask]
    .groupby(grp)["sales"]
    .agg(
        store_family_mean_sales="mean",
        store_family_median_sales="median",
        store_family_std_sales="std",
    )
    .reset_index()
)

store_enc = (
    full[train_mask]
    .groupby("store_nbr")["sales"]
    .agg(store_mean_sales="mean")
    .reset_index()
)

family_enc = (
    full[train_mask]
    .groupby("family")["sales"]
    .agg(family_mean_sales="mean")
    .reset_index()
)

full = full.merge(target_enc, on=grp,         how="left")
full = full.merge(store_enc,  on="store_nbr", how="left")
full = full.merge(family_enc, on="family",    how="left")

# Verhältnis Store/Family zum Gesamt-Mittelwert (relative Stärke)
global_mean = full[train_mask]["sales"].mean()
full["store_family_vs_global"] = full["store_family_mean_sales"] / (global_mean + 1e-6)
full["store_vs_global"]        = full["store_mean_sales"]        / (global_mean + 1e-6)
full["family_vs_global"]       = full["family_mean_sales"]       / (global_mean + 1e-6)




MemoryError: Unable to allocate 2.35 GiB for an array with shape (104, 3029400) and data type float64

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 13. Train / Test wieder trennen & speichern
# ─────────────────────────────────────────────────────────────────────────────
full = full.sort_values(["date", "store_nbr", "family"]).reset_index(drop=True)

test_ids = set(test["id"].tolist())
is_test  = full["id"].isin(test_ids)

train_fe    = full[~is_test].copy()
test_fe_out = full[is_test].copy()

# Validierungs-Split: letzte 15 Tage des Trains = Val
# (entspricht dem Test-Horizont von 16 Tagen)
val_cutoff   = train_fe["date"].max() - pd.Timedelta(days=15)
train_part   = train_fe[train_fe["date"] <= val_cutoff].copy()
val_part     = train_fe[train_fe["date"] >  val_cutoff].copy()

print(f"Train:      {train_part.shape}")
print(f"Val:        {val_part.shape}")
print(f"Test:       {test_fe_out.shape}")
print(f"Features:   {train_fe.shape[1]}")

# Speichern
train_part.to_csv(PROCESSED_DIR / "train_lagged_train_v2.csv", index=False)
val_part.to_csv(  PROCESSED_DIR / "train_lagged_val_v2.csv",   index=False)
test_fe_out.to_csv(PROCESSED_DIR / "test_lagged_v2.csv",       index=False)

print("\nGespeichert:")
print("  train_lagged_train_v2.csv")
print("  train_lagged_val_v2.csv")
print("  test_lagged_v2.csv")

# 

In [ ]:
# 14. Feature-Liste ausgeben (für 06_Modeling_v2)

EXCLUDE = ["id", "date", "sales"]

cat_cols = ["store_nbr", "family", "city", "state", "type", "cluster", "holiday_names"]
num_cols = [c for c in train_fe.columns if c not in cat_cols + EXCLUDE]
feature_cols = cat_cols + num_cols

print(f"\nAnzahl Features: {len(feature_cols)}")
print(feature_cols)